# Exercise 8: Testing, Validation, and Profiling

This notebook completes Sections **8.1 to 8.10** end-to-end.

It includes:
- Fully documented `stats.py` with NumPy-style docstrings and doctest examples.
- Defensive input validation and error handling.
- `unittest`, `pytest`, and Hypothesis property-based tests.
- Tolerance-based numeric checks with `numpy.testing.assert_allclose`.
- Coverage measurement to reach **100%** for `stats.py`.
- Static analysis with `pylint`.
- Performance profiling (`%timeit`, `cProfile`, `line_profiler`) for slow vs NumPy implementations.

In [1]:
from pathlib import Path
import importlib
import subprocess
import sys

WORKDIR = Path.cwd()
print(f"Working directory: {WORKDIR}")

# Ensure required packages exist; install only missing ones.
required = ["numpy", "pytest", "hypothesis", "coverage", "pylint", "line_profiler", "scalene"]
missing = []
for pkg in required:
    try:
        importlib.import_module(pkg)
    except ModuleNotFoundError:
        missing.append(pkg)

if missing:
    print("Installing missing packages:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
else:
    print("All required packages already installed.")

Working directory: c:\Users\nskin\Desktop\miniproject_computing\lect8ex
Installing missing packages: ['hypothesis', 'coverage', 'line_profiler', 'scalene']


## 8.1 and 8.2: Docstrings + Defensive Coding

`stats.py` now includes:
- A module-level docstring.
- Full NumPy-style docstrings for `mean`, `variance`, `std`, `normalize`.
- `Parameters`, `Returns`, `Raises`, and `Examples` sections.
- At least two doctest examples per function, including error examples.
- Defensive checks:
  - `mean`: empty data and NaN checks.
  - `variance`: `len(data) <= ddof` check.
  - `normalize`: zero standard deviation check.

In [2]:
stats_path = Path("stats.py")
if not stats_path.exists():
    raise FileNotFoundError("stats.py not found. Place this notebook in the same folder as stats.py.")

print(f"Found: {stats_path.resolve()}")
print("\nFirst 80 lines of stats.py:\n")
for i, line in enumerate(stats_path.read_text(encoding="utf-8").splitlines(), start=1):
    if i > 80:
        print("... (truncated)")
        break
    print(f"{i:>3}: {line}")

Found: C:\Users\nskin\Desktop\miniproject_computing\lect8ex\stats.py

First 80 lines of stats.py:

  1: """Basic statistical helpers with defensive validation.
  2: 
  3: This module provides small, dependency-light wrappers around common
  4: descriptive statistics used throughout the exercises: mean, variance,
  5: standard deviation, and z-score normalization.
  6: """
  7: 
  8: from __future__ import annotations
  9: 
 10: import numpy as np
 11: 
 12: 
 13: def mean(data: np.ndarray | list[float]) -> float:
 14:     """Compute the arithmetic mean of a 1D numeric dataset.
 15: 
 16:     Parameters
 17:     ----------
 18:     data : array-like of float
 19:         Input values. Any 1D array-like object accepted by
 20:         :func:`numpy.asarray` is supported.
 21: 
 22:     Returns
 23:     -------
 24:     float
 25:         Arithmetic mean of the input values.
 26: 
 27:     Raises
 28:     ------
 29:     ValueError
 30:         If ``data`` is empty.
 31:     ValueError
 32

## 8.3: Doctest

Run doctest directly against the examples in `stats.py` docstrings.

In [3]:
result = subprocess.run(
    [sys.executable, "-m", "doctest", "-v", "stats.py"],
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

Trying:
    mean([1, 2, 3, 4])
Expecting:
    2.5
ok
Trying:
    mean(np.array([-2.0, 0.0, 2.0]))
Expecting:
    0.0
ok
Trying:
    mean([])
Expecting:
    Traceback (most recent call last):
    ...
    ValueError: data must not be empty
ok
Trying:
    normalize([1, 2, 3]).tolist()
Expecting:
    [-1.224744871391589, 0.0, 1.224744871391589]
ok
Trying:
    normalize([10, 20]).tolist()
Expecting:
    [-1.0, 1.0]
ok
Trying:
    normalize([7, 7, 7])
Expecting:
    Traceback (most recent call last):
    ...
    ValueError: standard deviation must be non-zero
ok
Trying:
    std([1, 2, 3, 4])
Expecting:
    1.118033988749895
ok
Trying:
    std([2, 2, 2])
Expecting:
    0.0
ok
Trying:
    std([5], ddof=1)
Expecting:
    Traceback (most recent call last):
    ...
    ValueError: len(data) must be greater than ddof
ok
Trying:
    variance([1, 2, 3, 4], ddof=0)
Expecting:
    1.25
ok
Trying:
    variance([1, 2, 3, 4], ddof=1)
Expecting:
    1.6666666666666667
ok
Trying:
    variance([10, 10], ddo

## 8.4: Unittest

Run the `unittest` test suite (`test_stats_unittest.py`).

In [5]:
result = subprocess.run(
    [sys.executable, "-m", "unittest", "-v", "test_stats_unittest.py"],
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)


test_basic_mean (test_stats_unittest.TestMean.test_basic_mean) ... ok
test_identical_values (test_stats_unittest.TestMean.test_identical_values) ... ok
test_negative_numbers (test_stats_unittest.TestMean.test_negative_numbers) ... ok
test_raises_on_empty (test_stats_unittest.TestMean.test_raises_on_empty) ... ok
test_raises_on_nan (test_stats_unittest.TestMean.test_raises_on_nan) ... ok
test_mean_zero_std_one_after_normalization (test_stats_unittest.TestNormalize.test_mean_zero_std_one_after_normalization) ... ok
test_raises_on_empty (test_stats_unittest.TestNormalize.test_raises_on_empty) ... ok
test_raises_on_nan (test_stats_unittest.TestNormalize.test_raises_on_nan) ... ok
test_raises_when_identical (test_stats_unittest.TestNormalize.test_raises_when_identical) ... ok
test_square_relationship (test_stats_unittest.TestStd.test_square_relationship) ... ok
test_error_when_ddof_gte_len (test_stats_unittest.TestVariance.test_error_when_ddof_gte_len) ... ok
test_population_variance (test

## 8.5: Pytest

Run the pytest rewrite in `test_stats_pytest.py`.

This file includes:
- Plain `assert` statements.
- `pytest.approx` for float comparisons.
- `@pytest.mark.parametrize` with 4 datasets for `mean`.
- `pytest.raises` checks for all error branches.

In [6]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "test_stats_pytest.py"],
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

..................                                                       [100%]
18 passed in 3.10s



## 8.6: Property-based Testing with Hypothesis

Properties checked automatically:
- Shift invariance of mean.
- Variance scaling law.
- Normalization outputs mean ≈ 0 and std ≈ 1.

In [7]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "test_stats_hypothesis.py"],
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

...                                                                      [100%]
3 passed in 2.74s



## 8.7: Tolerance-based Testing

Using `numpy.testing.assert_allclose` and strict absolute tolerances for normalization moments.

In [8]:
import numpy as np
from numpy.testing import assert_allclose

from stats import mean, normalize, std, variance

assert_allclose(mean([1, 2, 3, 4, 5]), 3.0, rtol=1e-12)

rng = np.random.default_rng(42)
data = rng.normal(0, 2, 10000)
assert_allclose(variance(data, ddof=1), 4.0, rtol=0.05)

normalized = normalize(data)
assert abs(mean(normalized)) < 1e-12
assert abs(std(normalized) - 1.0) < 1e-12

print("Tolerance-based checks passed.")

Tolerance-based checks passed.


## 8.8: Code Coverage

Run the exact command sequence requested and inspect missing lines.

Goal: 100% coverage on `stats.py`.

In [9]:
run_cov = subprocess.run(
    [sys.executable, "-m", "coverage", "run", "-m", "pytest", "test_stats_pytest.py"],
    capture_output=True,
    text=True,
    check=True,
)
print(run_cov.stdout)

report_cov = subprocess.run(
    [sys.executable, "-m", "coverage", "report", "-m"],
    capture_output=True,
    text=True,
    check=True,
)
print(report_cov.stdout)

============================= test session starts =============================
platform win32 -- Python 3.13.9, pytest-8.4.2, pluggy-1.5.0
rootdir: c:\Users\nskin\Desktop\miniproject_computing\lect8ex
plugins: anyio-4.10.0, hypothesis-6.151.10
collected 18 items

test_stats_pytest.py ..................                                  [100%]

============================= 18 passed in 1.29s ==============================

Name                   Stmts   Miss  Cover   Missing
----------------------------------------------------
stats.py                  25      0   100%
test_stats_pytest.py      52      0   100%
----------------------------------------------------
TOTAL                     77      0   100%



## 8.9: Static Code Analysis with pylint

Run pylint on:
- `messy_solver.py` (refactored target score >= 9.0/10)
- `stats.py`

In [10]:
result = subprocess.run(
    [sys.executable, "-m", "pylint", "messy_solver.py", "stats.py"],
    capture_output=True,
    text=True,
    check=False,
)
print(result.stdout)
print(result.stderr)

if result.returncode not in (0, 4, 8, 16, 24, 28, 32):
    raise RuntimeError("Unexpected pylint return code")


------------------------------------
Your code has been rated at 10.00/10





## 8.10: Profiling

Compare pure-Python and NumPy implementations in `perf_compare.py`.

Expected trend: NumPy is substantially faster for large arrays.

In [11]:
import cProfile
import pstats
import io

import numpy as np

from perf_compare import mean_fast, mean_slow, variance_fast, variance_slow

N = 1_000_000
rng = np.random.default_rng(42)
data_np = rng.normal(0, 1, N)
data_py = data_np.tolist()

print(f"Dataset size: {N:,}")

Dataset size: 1,000,000


In [12]:
# Baseline speed comparison with %timeit
%timeit mean_slow(data_py)
%timeit mean_fast(data_np)
%timeit variance_slow(data_py)
%timeit variance_fast(data_np)

73.9 ms ± 1.18 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
698 μs ± 12.2 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
158 ms ± 7.01 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
6.3 ms ± 230 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [13]:
# cProfile summary for mean_slow
profiler = cProfile.Profile()
profiler.enable()
mean_slow(data_py)
profiler.disable()

stream = io.StringIO()
stats_obj = pstats.Stats(profiler, stream=stream).sort_stats("cumtime")
stats_obj.print_stats(10)
print(stream.getvalue())

         48 function calls (47 primitive calls) in 0.085 seconds

   Ordered by: cumulative time
   List reduced from 21 to 10 due to restriction <10>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      3/2    0.000    0.000    0.085    0.042 c:\Users\nskin\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3663(run_code)
        2    0.000    0.000    0.085    0.042 {built-in method builtins.exec}
        1    0.080    0.080    0.085    0.085 C:\Users\nskin\AppData\Local\Temp\ipykernel_14524\1799176112.py:1(<module>)
        1    0.005    0.005    0.005    0.005 c:\Users\nskin\Desktop\miniproject_computing\lect8ex\perf_compare.py:8(mean_slow)
        2    0.000    0.000    0.000    0.000 c:\Users\nskin\anaconda3\Lib\codeop.py:113(__call__)
        2    0.000    0.000    0.000    0.000 {built-in method builtins.compile}
        1    0.000    0.000    0.000    0.000 {method 'disable' of '_lsprof.Profiler' objects}
        2    0.000    0.000    0.000

In [14]:
# line_profiler: pinpoint hotspot line(s)
%load_ext line_profiler
%lprun -f mean_slow mean_slow(data_py)
%lprun -f variance_slow variance_slow(data_py)

Timer unit: 1e-07 s

Total time: 2.43965 s
File: c:\Users\nskin\Desktop\miniproject_computing\lect8ex\perf_compare.py
Function: variance_slow at line 28

Line #      Hits         Time  Per Hit   % Time  Line Contents
    28                                           def variance_slow(data: list[float], ddof: int = 0) -> float:
    29                                               """Compute variance in pure Python."""
    30         1         12.0     12.0      0.0      n = len(data)
    31         1          7.0      7.0      0.0      if n <= ddof:
    32                                                   raise ValueError("len(data) must be greater than ddof")
    33                                           
    34         1   11458791.0 1.15e+07     47.0      mu = mean_slow(data)
    35         1         21.0     21.0      0.0      total = 0.0
    36   1000001    4306080.0      4.3     17.7      for value in data:
    37   1000000    4276014.0      4.3     17.5          diff = value - 

## Summary

If all cells pass:
- Docstrings and defensive coding are complete.
- Doctest, unittest, pytest, and Hypothesis all pass.
- Coverage is 100% on `stats.py`.
- pylint score is >= 9.0/10 (target achieved: 10.00/10).
- Profiling demonstrates the NumPy speedup and identifies pure-Python bottlenecks.